# AlexNet — ImageNet Classification with Deep Convolutional Neural Networks

**Citation:** Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *Advances in Neural Information Processing Systems (NeurIPS)*, 25, 1097–1105.

**Registered names:** `alexnet`

**Summary:** AlexNet won ILSVRC 2012 by a 10.9% margin, demonstrating that deep CNNs trained with ReLU activations, dropout regularisation, and data augmentation could dramatically outperform hand-crafted features. It ignited the deep learning revolution in computer vision.

---

| Property | Value |
|----------|-------|
| Parameters | ~61M |
| embed_dim | 4096 |
| feature_dims | [64, 192, 384, 256, 256] |
| Top-1 (ImageNet) | 56.5% |
| Year | 2012 |

## Architecture Diagram

```
Input [B, 3, H, W]
       │
       ▼
┌──────────────────────────────────────────────┐
│ Stage 0  Conv(11×11, s4) + ReLU + MaxPool    │ → [B, 64,  H/8,  W/8]
└──────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────┐
│ Stage 1  Conv(5×5, pad2) + ReLU + MaxPool    │ → [B, 192, H/16, W/16]
└──────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────┐
│ Stage 2  Conv(3×3, pad1) + ReLU             │ → [B, 384, H/16, W/16]
└──────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────┐
│ Stage 3  Conv(3×3, pad1) + ReLU             │ → [B, 256, H/16, W/16]
└──────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────┐
│ Stage 4  Conv(3×3, pad1) + ReLU + MaxPool    │ → [B, 256, H/32, W/32]
└──────────────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────────────┐
│ AdaptiveAvgPool(6×6) → Flatten               │ → [B, 9216]
│ Dropout → FC(4096) → ReLU                   │
│ Dropout → FC(4096) → ReLU                   │
└──────────────────────────────────────────────┘
       │
       ▼
pooled_output [B, 4096]
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

colors = ['#4a9eff', '#6bcb77', '#ff9f43', '#ff6b9d', '#a29bfe', '#fd79a8']
stages = [
    ('Input\n3×H×W', '#555566', 0.5, 3.5),
    ('Stage 0\nConv 11×11 s4\nReLU+MaxPool\n64ch H/8', colors[0], 2.2, 3.5),
    ('Stage 1\nConv 5×5 pad2\nReLU+MaxPool\n192ch H/16', colors[1], 4.0, 3.5),
    ('Stage 2\nConv 3×3\nReLU\n384ch', colors[2], 5.8, 3.5),
    ('Stage 3\nConv 3×3\nReLU\n256ch', colors[3], 7.6, 3.5),
    ('Stage 4\nConv 3×3\nReLU+MaxPool\n256ch H/32', colors[4], 9.4, 3.5),
    ('FC Head\nPool→9216\nFC×2→4096', colors[5], 11.5, 3.5),
]

for label, color, cx, cy in stages:
    box = FancyBboxPatch((cx - 0.7, cy - 1.1), 1.4, 2.2,
                          boxstyle='round,pad=0.08', facecolor=color,
                          edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.add_patch(box)
    ax.text(cx, cy, label, ha='center', va='center', fontsize=7.5,
            color='white', fontweight='bold', linespacing=1.4)

# Arrows between stages
xs = [stage[2] for stage in stages]
for i in range(len(xs) - 1):
    ax.annotate('', xy=(xs[i+1] - 0.72, 3.5), xytext=(xs[i] + 0.72, 3.5),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))

# Feature map size annotations below
sizes = ['3×H×W', '64×H/8', '192×H/16', '384×H/16', '256×H/16', '256×H/32', '4096']
for i, (size, (_, _, cx, _)) in enumerate(zip(sizes, stages)):
    ax.text(cx, 1.8, size, ha='center', va='center', fontsize=7,
            color='#aaaaaa', style='italic')

ax.text(7, 6.5, 'AlexNet Architecture (Krizhevsky et al. 2012)',
        ha='center', va='center', fontsize=13, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

## Key Innovations

### 1. ReLU Activations
AlexNet used **ReLU** (Rectified Linear Unit) instead of sigmoid/tanh:
$$f(x) = \max(0, x)$$
ReLU trains ~6× faster on CIFAR-10 than tanh because it doesn't saturate for large positive activations, avoiding the vanishing gradient problem.

### 2. Dropout Regularisation
Applied after each FC layer with $p = 0.5$:
$$\tilde{h} = h \odot \text{Bernoulli}(1-p)$$
Forces the network to learn redundant representations; interpretable as training an ensemble of $2^N$ thinned networks.

### 3. GPU Training (Dual GTX 580)
The original paper split the model across two GPUs due to 3 GB memory constraints. Modern reimplementations run on a single GPU.

### 4. Local Response Normalisation (LRN)
Originally used between conv stages (inspired by lateral inhibition in neuroscience). Modern implementations replace this with BatchNorm.

### 5. Data Augmentation
Random crops + horizontal flips + colour jitter — now standard practice.

### Historical Significance
AlexNet reduced the ILSVRC 2012 top-5 error from 26.2% (prior winner) to **15.3%**, a margin so large it convinced the research community that deep learning was the path forward.

## Code Walkthrough

The backbone is implemented in `hale_core/nn/vision/cnn/alexnet.py` as `AlexNet(BaseVisionBackbone)`.

**Stage construction:**
```python
self.stage0 = nn.Sequential(
    nn.Conv2d(in_chans, 64, kernel_size=11, stride=4, padding=2),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=3, stride=2),
)
```
The large 11×11 kernel was needed in 2012 to capture texture at coarse scale. Modern designs use 3×3 stacks instead.

**AdaptiveAvgPool** normalises spatial size to 6×6 before the FC layers, making the backbone input-resolution-agnostic:
```python
self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
```

**Weight init:** Kaiming normal for conv (correct variance under ReLU), trunc_normal(std=0.01) for FC layers (matches original paper).

**BackboneOutput:** Returns 5 feature maps (one per stage) + 4096-dim pooled embedding from FC layers.

In [ ]:
import torch
import sys
sys.path.insert(0, '../../src')

from hale_core.nn.vision.registry import build_vision_backbone

# Build AlexNet
bb = build_vision_backbone('alexnet', img_size=224)
bb.eval()

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = bb(x)

print(f'pooled_output: {out.pooled_output.shape}')  # [1, 4096]
print(f'patch_tokens:  {out.patch_tokens}')          # None
print(f'feature_maps:')
for i, fm in enumerate(out.feature_maps):
    print(f'  stage {i}: {tuple(fm.shape)}')

print(f'\nParameters: {bb.num_parameters():,}')
print(f'embed_dim:   {bb.embed_dim}')
print(f'feature_dims: {bb.feature_dims}')

## References

| Paper | Venue | Key Contribution |
|-------|-------|------------------|
| Krizhevsky et al. (2012). ImageNet Classification with Deep CNNs. | NeurIPS 2012 | AlexNet — ReLU, Dropout, GPU training |
| LeCun et al. (1989). Backpropagation to handwritten digit recognition. | Neural Computation | LeNet — Pioneering CNN |
| Simonyan & Zisserman (2015). Very Deep CNNs. | ICLR 2015 | VGGNet — small 3×3 filters, more depth |
| He et al. (2016). Deep Residual Learning. | CVPR 2016 | ResNet — skip connections, 1000+ layers |
| Srivastava et al. (2014). Dropout. | JMLR 2014 | Formal analysis of dropout regularisation |

**Full Citation:**
> Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. In *Advances in Neural Information Processing Systems* (Vol. 25, pp. 1097–1105). Curran Associates.